In [1]:
import os
import datetime
import argparse
import logging
import numpy as np

import torch
import torch.distributed as dist
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    BertModel,
    get_linear_schedule_with_warmup,
)
from transformers.models.bert.modeling_bert import (
    BertLayer,
    BertEmbeddings,
    BertPooler,
    BertEncoder,
)
from datasets import load_dataset, load_from_disk
from tqdm import tqdm
from utils.buffer import TensorBuffer

from train import PolarTrainer, process_group_setup


/home/aerith/anaconda3/envs/polar-sgd/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

parser = argparse.ArgumentParser()
parser.add_argument(
    "--model", type=str, default="bert-base-uncased", help="model name"
)
parser.add_argument(
    "--pretrained", type=bool, default=False, help="use pretrained model"
)
parser.add_argument(
    "--data_path",
    type=str,
    default="data/glue/sst2",
    help="path to the training data",
)
parser.add_argument(
    "--model_path",
    type=str,
    default="models/bert-base-uncased",
    help="path to the model",
)
parser.add_argument(
    "--tokenizer_path",
    type=str,
    default="tokenizer/bert-base-uncased",
    help="path to the tokenizer",
)
parser.add_argument(
    "--num_labels", type=int, default=2, help="classification kinds"
)
parser.add_argument(
    "--max_length", type=int, default=128, help="max length of the input sequence"
)
parser.add_argument("--batch_size", type=int, default=16)
parser.add_argument("--lr", type=float, default=1e-5)
parser.add_argument("--epochs", type=int, default=3)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--local_steps", type=int, default=4, help="local steps")

args = parser.parse_args()

global_group, inter_group, local_group = process_group_setup()

config = AutoConfig.from_pretrained(
    "models/bert-base-uncased", torch_dtype=torch.float16, num_labels=2
)
model = AutoModelForSequenceClassification.from_config(config)

usage: ipykernel_launcher.py [-h] [--model MODEL] [--pretrained PRETRAINED]
                             [--data_path DATA_PATH] [--model_path MODEL_PATH]
                             [--tokenizer_path TOKENIZER_PATH]
                             [--num_labels NUM_LABELS]
                             [--max_length MAX_LENGTH]
                             [--batch_size BATCH_SIZE] [--lr LR]
                             [--epochs EPOCHS] [--seed SEED]
                             [--local_steps LOCAL_STEPS]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1000/jupyter/runtime/kernel-v36e15024ff6ccc8c3dc9803ac2c59a9297f5a1c0b.json


SystemExit: 2

In [5]:
config = AutoConfig.from_pretrained(
    "models/bert-base-uncased", torch_dtype=torch.float16, num_labels=2
)
model = AutoModelForSequenceClassification.from_config(config)

In [6]:
def get_bert_all_layers(module: torch.nn.Module):
    # 遍历模块的所有子模块
    for child in module.modules():
        if isinstance(child, (BertModel, BertEncoder, torch.nn.ModuleList)):
            continue  # 跳过容器类模块，避免重复处理
        if isinstance(child, (BertEmbeddings, BertLayer, BertPooler, torch.nn.Linear, torch.nn.Dropout)):
            yield child

for layer in get_bert_all_layers(model):
    print(f"Layer: {layer.__class__.__name__}, Parameters: {sum(p.numel() for p in layer.parameters())}")

Layer: BertEmbeddings, Parameters: 23837184
Layer: Dropout, Parameters: 0
Layer: BertLayer, Parameters: 7087872
Layer: Linear, Parameters: 590592
Layer: Linear, Parameters: 590592
Layer: Linear, Parameters: 590592
Layer: Dropout, Parameters: 0
Layer: Linear, Parameters: 590592
Layer: Dropout, Parameters: 0
Layer: Linear, Parameters: 2362368
Layer: Linear, Parameters: 2360064
Layer: Dropout, Parameters: 0
Layer: BertLayer, Parameters: 7087872
Layer: Linear, Parameters: 590592
Layer: Linear, Parameters: 590592
Layer: Linear, Parameters: 590592
Layer: Dropout, Parameters: 0
Layer: Linear, Parameters: 590592
Layer: Dropout, Parameters: 0
Layer: Linear, Parameters: 2362368
Layer: Linear, Parameters: 2360064
Layer: Dropout, Parameters: 0
Layer: BertLayer, Parameters: 7087872
Layer: Linear, Parameters: 590592
Layer: Linear, Parameters: 590592
Layer: Linear, Parameters: 590592
Layer: Dropout, Parameters: 0
Layer: Linear, Parameters: 590592
Layer: Dropout, Parameters: 0
Layer: Linear, Parameter

In [ ]:
def get_bert_all_layers_2(module: torch.nn.Module):
    for name, child in module.named_children():
        if isinstance(child, BertModel):
            yield from get_bert_all_layers(child)
        if isinstance(child, BertEncoder):
            yield from get_bert_all_layers(child)
        if isinstance(child, torch.nn.ModuleList):
            yield from get_bert_all_layers(child)
        if isinstance(child, BertEmbeddings):
            yield child
        if isinstance(child, BertLayer):
            yield child
        if isinstance(child, BertPooler):
            yield child
        if isinstance(child, torch.nn.Linear):
            yield child
        if isinstance(child, torch.nn.Dropout):
            yield child
            
for layer in get_bert_all_layers_2(model):
    print(f"Layer: {layer.__class__.__name__}, Parameters: {sum(p.numel() for p in layer.parameters())}")

In [4]:
def get_bert_all_layers(module: torch.nn.Module):
    for name, child in module.named_children():
        if isinstance(child, BertModel):
            yield from get_bert_all_layers(child)
        if isinstance(child, BertEncoder):
            yield from get_bert_all_layers(child)
        if isinstance(child, torch.nn.ModuleList):
            yield from get_bert_all_layers(child)
        if isinstance(child, BertEmbeddings):
            yield child
        if isinstance(child, BertLayer):
            yield child
        if isinstance(child, BertPooler):
            yield child
        if isinstance(child, torch.nn.Linear):
            yield child
        if isinstance(child, torch.nn.Dropout):
            yield child

def split_bert_based_model_into_partitions(model, num_partitions):
    # TODO: the module list need to reverse.
    modules = list(get_bert_all_layers(model))
    layer_param_sizes = []

    for layer in modules:
        layer_param_sizes.append(
            sum(p.numel() for p in layer.parameters() if p.requires_grad)
        )

    total_params = sum(layer_param_sizes)
    target_size = total_params / num_partitions

    partitions = []
    i = 0
    while True:
        partitions_params_size = 0
        partition = []
        while partitions_params_size < target_size and i < len(layer_param_sizes):
            partitions_params_size += layer_param_sizes[i]
            partition.append(modules[i])
            i += 1

        if i == len(layer_param_sizes):
            partitions.append(partition)
            break

        if abs(partitions_params_size - target_size) < abs(
            partitions_params_size + layer_param_sizes[i] - target_size
        ):
            partitions.append(partition)
        else:
            partitions.append(partition.append(modules[i]))
            i += 1

    return partitions

In [5]:
partitions = split_bert_based_model_into_partitions(model, 4)

In [ ]:
for partition in partitions:
    for layer in partition:
        for p in layer.parameters():
            print(type(p.data))

<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.paramete

In [10]:
def flattened(grad_partition: list[torch.Tensor]):
    flattend = torch.cat([t.flatten() for t in grad_partition])
    original_shape = [t.shape for t in grad_partition]
    original_size = [t.numel() for t in grad_partition]
    
    return flattend, original_shape, original_size

def reshape(flattened_grad_pred: torch.Tensor, original_shape, original_size):
    restored_tensors =  torch.split(flattened_grad_pred, original_size)
    restored_tensors = [t.view(shape) for t, shape in zip(restored_tensors, original_shape)]

    return restored_tensors

In [ ]:
flattened_grad_pred, original_shape, original_size = flattened(grads[0])  
            